# Notebook Overview — Validate Autoencoder Video Representations

## Purpose

This notebook loads and validates the standardized autoencoder representations generated by Notebook 03 for downstream representation-based VideoQA experiments.

Rather than training the autoencoder or generating new latent embeddings, the notebook restores the trained model and the existing segment-level and video-level representation files, standardizes the video representation schema, verifies the train, validation, and test representation records, and generates a representation summary report.

The resulting `autoencoder_video` representation artifact is consumed by Notebook 07, which selects the required training and validation representations and merges them with the corresponding NExT-QA annotations and shared CLIP text embeddings.

## Inputs

* Trained autoencoder model generated by Notebook 03
* Autoencoder segment-level latent representations generated by Notebook 03
* Autoencoder video-level latent representations generated by Notebook 03
* NExT-QA annotations
* Shared project configuration

## Outputs

* Standardized autoencoder video representation file
* Validated segment-level and video-level representation data
* Autoencoder representation summary report
* Notebook 07-compatible representation artifacts

## Processing Workflow

1. Initialize the project environment.
2. Configure the autoencoder representation-validation workflow.
3. Verify the runtime environment.
4. Load the trained autoencoder model.
5. Prepare a development evaluation dataset for configuration and annotation validation.
6. Load and standardize the existing segment-level and video-level autoencoder representations.
7. Validate the standardized representation schema and embedding values.
8. Verify the loaded representation artifacts.
9. Generate and save the autoencoder representation summary report.
10. Display representative video representation records.
11. Summarize the completed representation-validation workflow.

## Downstream Consumer

Notebook 07 — Run Representation-Based VideoQA


### 🔷 Step 0 — Configure Experiment Execution

* Configure the experiment settings used throughout the notebook before execution begins.
* Specify EXPERIMENT_NAME, which identifies the experiment and determines the corresponding output directory.
* Select the prediction method for representation-based VideoQA experiments (Notebook 07 only) and ensure it matches the value specified in EXPERIMENT_NAME.
* Choose whether to evaluate the full validation split or a development subset and specify the development subset size when applicable.
* Configure notebook runtime options, including GPU requirements and verbose progress reporting.
* Keep these settings consistent across all notebooks that participate in the same experiment.

In [ ]:
# ============================================================
# Step 0: Configure Experiment Execution
# ============================================================
# Review these settings before running the notebook.
# Keep these values consistent across all notebooks in the experiment.
# ------------------------------------------------------------
#
# EXPERIMENT_NAME identifies the experiment and output directory.
#
# Format:
#   <representation>_<prediction_method>_<dataset>
#
# Representation:
#   qwen2vl
#   clip
#   ae_seg6s_stride4
#   hybrid_clip_ae
#
# Prediction method:
#   baseline
#   similarity
#   mlp
#   interaction
#   gated
#   bilinear
#
# Dataset:
#   dev100
#   dev500
#   full
#
# Examples:
#   qwen2vl_baseline_dev100
#   clip_bilinear_full
#   ae_seg6s_stride4_mlp_dev100
#   hybrid_clip_ae_bilinear_dev500
#
EXPERIMENT_NAME = "ae_seg6s_stride4_mlp_dev100"

# Prediction method (used only by Notebook 07).
# Must match the prediction method specified in EXPERIMENT_NAME.
# Supported values:
#   cosine_similarity
#   fusion_mlp_classifier
#   interaction_fusion_classifier
#   gated_fusion_classifier
#   bilinear_fusion_classifier
#
REPRESENTATION_VIDEOQA_METHOD = "fusion_mlp_classifier"

# Evaluate the full validation split if True.
RUN_FULL_EVALUATION_SPLIT = False

# Development subset size (typically 100 or 500).
DEVELOPMENT_SUBSET_SIZE = 100

# Require an NVIDIA L4 GPU.
REQUIRE_L4_GPU = True

# Display detailed notebook progress.
VERBOSE = True

# Save generated artifacts to Google Drive.
# Disabled by default for the public tutorial.
ENABLE_GOOGLE_DRIVE_WRITES = False



### 🔷 Step 1 — Initialize Environment for Autoencoder Representation Preparation

* Initialize the notebook runtime and prepare the project execution environment.
* Mount Google Drive and restore required project resources.
* Clone the public GitHub repository using sparse checkout without requiring authentication.
* Verify that the required datasets and experiment directories are available.
* Confirm readiness for autoencoder representation preparation.



In [ ]:
# ============================================================
# Step 1: Initialize Environment for Autoencoder Representation Preparation
# ============================================================

# These notebook-level controls enable detailed reporting, enforce the expected
# accelerator requirement, and define the known size of the NExT-QA video set.

EXPECTED_NEXTQA_VIDEO_COUNT = 5440

import os
import time
from pathlib import Path

import pandas as pd

from google.colab import drive

print("Initializing notebook environment...")
print("-" * 60)

# ------------------------------------------------------------
# DRIVE MOUNT (REQUIRED)
# ------------------------------------------------------------

# Google Drive stores the trained autoencoder and experiment artifacts produced
# by Notebook 03, so it must be available before those resources are restored.
GOOGLE_DRIVE_MOUNT = "/content/drive"

if not os.path.exists(GOOGLE_DRIVE_MOUNT):
    print("Mounting Google Drive...")
    drive.mount(GOOGLE_DRIVE_MOUNT)
else:
    print("Google Drive already mounted.")

# ------------------------------------------------------------
# REPO CONFIG
# ------------------------------------------------------------

# Define the local Colab location where the project repository will be cloned
# or reused during this notebook session.
REPO_NAME = "videoqa-representation-comparison"
REPO_OWNER = "pgailinas"
REPO_BASE_DIR = "/content"
REPO_DIR = os.path.join(REPO_BASE_DIR, REPO_NAME)

# Build the public repository URL.
repo_url = (
    f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
)

os.chdir(REPO_BASE_DIR)

# A sparse checkout downloads only the directories required by this notebook,
# reducing startup time and unnecessary storage use in the Colab runtime.
if not os.path.exists(REPO_DIR):

    print("Cloning project repository...")

    !git clone --quiet --filter=blob:none --no-checkout {repo_url}

    os.chdir(REPO_DIR)

    !git sparse-checkout init --cone
    !git sparse-checkout set src datasets outputs
    !git checkout --quiet main

else:

    # Reuse the existing checkout when the cell is rerun in the same session.
    print("Project repository already available.")
    os.chdir(REPO_DIR)

print(f"Repository ready: {REPO_DIR}")

# ------------------------------------------------------------
# LOAD PROJECT MODULES
# ------------------------------------------------------------

# Import the shared project configuration first because later modules and path
# checks depend on the constants and helpers it defines.
print("\nLoading project configuration and utility modules...")

from src.videoqa_representation_config import *

# ------------------------------------------------------------
# NOTEBOOK-SPECIFIC EXPERIMENT SELECTION
# ------------------------------------------------------------

# Apply the selected experiment configuration before referencing its paths.
configure_experiment(EXPERIMENT_NAME)


# NOTE: KEEP (needed for evaluation)
# These modules provide artifact restoration, the trained model definition,
# and validation utilities needed to interpret Notebook 03 artifacts.
from src.videoqa_project_restore import restore_project_artifacts
from src.training_validation import *
from src.autoencoder_model import *

# ------------------------------------------------------------
# OPTIONAL DATA MODULES (Notebook 04 SAFE SET)
# ------------------------------------------------------------

# Notebook 04 requires annotation metadata for split and coverage validation,
# but does not need to load or decode the source video collection.
from src.nextqa_metadata import *

# ------------------------------------------------------------
# OUTPUT SETUP
# ------------------------------------------------------------

# Ensure the configured output root exists before checking the complete set of
# repository directories required by the remaining notebook steps.
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

required_paths = [
    Path("src"),
    Path("datasets"),
    OUTPUTS_DIR,
]

missing_paths = [p for p in required_paths if not p.exists()]

# Fail early when the repository checkout or configuration is incomplete,
# preventing later errors from being mistaken for representation-data issues.
if missing_paths:
    for p in missing_paths:
        print(f"Missing required path: {p}")

    raise FileNotFoundError("One or more required project paths are missing.")

print("Configuration loaded.")
print("Project paths initialized.")

# ------------------------------------------------------------
# RESTORE LOCAL PROJECT ARTIFACTS
# ------------------------------------------------------------

print("\nChecking local VideoQA project artifacts...")

# Restore the Notebook 03 model and experiment artifacts.
restore_project_artifacts(
    drive_archive_path=PROJECT_ARTIFACTS_DRIVE_ARCHIVE,
    local_archive_path=PROJECT_ARTIFACTS_LOCAL_ARCHIVE,
    project_dir=VIDEOQA_PROJECT_DIR,
    required_paths=[
        AUTOENCODER_MODEL_PATH,
    ],
    verbose=VERBOSE,
)

print("Local VideoQA project artifacts ready.")

# ------------------------------------------------------------
# LOAD ONLY ANNOTATIONS
# ------------------------------------------------------------

# Only annotation tables are needed here to establish authoritative split
# membership and evaluation labels; the raw videos are intentionally not loaded.
print("\nLoading NExT-QA annotations (evaluation labels only)...")

split_annotations = load_nextqa_split_annotations(
    annotations_dir=QUESTIONS_DIR,
    verbose=VERBOSE,
)

# Combine the individual train, validation, and test tables into one consistent
# dataframe for subsequent representation coverage checks.
annotations_df = combine_nextqa_annotations(
    split_dataframes=split_annotations,
    verbose=VERBOSE,
)

# Summarize the combined annotations now so the expected dataset composition is
# visible before generated autoencoder artifacts are inspected.
split_summary_df = summarize_nextqa_splits(
    annotations=annotations_df,
)

print("\nDataset metadata loaded.")
print(f"Annotation records: {len(annotations_df):,}")

# ------------------------------------------------------------
# VERIFY AUTOENCODER MODEL
# ------------------------------------------------------------

print("\nChecking Notebook 03 autoencoder model...")

# The checkpoint is a required upstream dependency; validation cannot proceed
# meaningfully when the corresponding trained model is absent.
if not AUTOENCODER_MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Missing trained autoencoder model: {AUTOENCODER_MODEL_PATH}"
    )

print("Notebook 03 autoencoder model found.")

# Display the annotation split summary in verbose mode as an initial reference
# for the representation counts examined later in the notebook.
if VERBOSE:
    print("\nSplit Summary")
    print("-" * 60)
    display(split_summary_df)

print("\nEnvironment initialization complete.")
print("-" * 60)
print("Notebook ready for embedding extraction.")



### 🔷 Step 2 — Define Autoencoder Representation Configuration

* Define the configuration used to load and validate the autoencoder representation artifacts.
* Use the active experiment configuration established during environment initialization.
* Configure the evaluation split, development subset size, and randomization settings.
* Record the autoencoder frame, segment-sampling, and batch-size settings associated with the experiment.
* Define inference-only operation for the representation workflow.
* Display and validate the active representation configuration.




In [ ]:
# ============================================================
# Step 2: Define Autoencoder Representation Configuration
# ============================================================

print("\nDefining autoencoder representation configuration...\n")

# ------------------------------------------------------------
# Import shared constants (single source of truth)
# ------------------------------------------------------------

# Reuse the project-wide constants so representation extraction remains aligned
# with the experiment settings used to train the autoencoder in Notebook 03.
from src.videoqa_representation_config import *

# ------------------------------------------------------------
# Representation Extraction Configuration
# ------------------------------------------------------------

# Collect the extraction controls in one dictionary so later steps can reference
# a single, explicit record of the dataset, model, and output settings.
REPRESENTATION_CONFIG = {
    # Dataset control
    "evaluation_split": EVALUATION_SPLIT,
    "development_subset_size": DEVELOPMENT_SUBSET_SIZE,
    "random_seed": RANDOM_SEED,

    # Autoencoder representation settings
    "frame_size": AUTOENCODER_FRAME_SIZE,
    "frames_per_segment": AUTOENCODER_FRAMES_PER_SEGMENT,
    "batch_size": AUTOENCODER_BATCH_SIZE,

    # Model usage mode (STRICTLY INFERENCE ONLY)
    # This notebook extracts latent vectors from an already trained model and
    # therefore must not perform optimization or update model parameters.
    "inference_mode": "autoencoder_latent_extraction",

    # Output control
    # Saving latent vectors makes the validated representations available to
    # downstream VideoQA experiments without rerunning the autoencoder.
    "save_latent_vectors": True,
    "verbose": True,
}

# ------------------------------------------------------------
# Validate Configuration
# ------------------------------------------------------------

# Fail immediately if any required numeric setting is invalid, since zero or
# negative values would make frame preparation or batched extraction impossible.
assert REPRESENTATION_CONFIG["development_subset_size"] > 0
assert REPRESENTATION_CONFIG["frame_size"] > 0
assert REPRESENTATION_CONFIG["frames_per_segment"] > 0
assert REPRESENTATION_CONFIG["batch_size"] > 0

# ------------------------------------------------------------
# Display Configuration
# ------------------------------------------------------------

# Print the complete configuration to make the extraction conditions visible
# and reproducible before model loading or representation processing begins.
print("Representation Extraction Configuration:")
for key, value in REPRESENTATION_CONFIG.items():
    print(f"  {key:<30}: {value}")



### 🔷 Step 3 — Verify Runtime Environment and Dependencies

* Display Python, platform, PyTorch, and CUDA runtime information.
* Verify GPU availability and enforce the configured L4 GPU requirement when enabled.
* Display detected GPU hardware and memory information.
* Verify required Python package availability.
* Confirm runtime readiness before loading the trained autoencoder and representation files.


In [ ]:
# ============================================================
# Step 3: Verify Runtime Environment and Dependencies
# ============================================================

import sys
import platform
import importlib

print("Verifying runtime environment and dependencies...\n")

# ------------------------------------------------------------
# Runtime Information
# ------------------------------------------------------------

# Record the interpreter and operating-system environment so later execution
# results can be traced to the exact Colab runtime configuration.
print("Runtime Information")
print("-" * 60)
print(f"Python Version : {sys.version.split()[0]}")
print(f"Platform       : {platform.platform()}")

# ------------------------------------------------------------
# PyTorch / CUDA Verification
# ------------------------------------------------------------

# Default to CPU until PyTorch confirms that a compatible CUDA device is
# available and satisfies the notebook's hardware requirement.
device = "cpu"

try:
    import torch

    print("\nPyTorch Information")
    print("-" * 60)
    print(f"PyTorch Version : {torch.__version__}")
    print(f"CUDA Available  : {torch.cuda.is_available()}")

    if torch.cuda.is_available():

        # Report every visible GPU because hosted runtimes may expose more than
        # one device or provide hardware that differs between sessions.
        print(f"CUDA Version    : {torch.version.cuda}")
        print(f"GPU Count       : {torch.cuda.device_count()}")

        for idx in range(torch.cuda.device_count()):

            gpu_name = torch.cuda.get_device_name(idx)
            gpu_props = torch.cuda.get_device_properties(idx)

            # Convert the device memory from bytes to gigabytes for a more
            # readable indication of the available accelerator capacity.
            total_memory_gb = gpu_props.total_memory / (1024 ** 3)

            print(f"GPU {idx}          : {gpu_name}")
            print(f"GPU {idx} Memory   : {total_memory_gb:.1f} GB")

        # Current allocation statistics help identify whether an earlier cell
        # has already consumed GPU memory before representation processing.
        allocated_gb = torch.cuda.memory_allocated() / (1024 ** 3)
        reserved_gb = torch.cuda.memory_reserved() / (1024 ** 3)

        print(f"Allocated Memory : {allocated_gb:.2f} GB")
        print(f"Reserved Memory  : {reserved_gb:.2f} GB")

        # The first CUDA device is treated as the primary execution target and
        # is checked against the experiment's expected L4 hardware.
        primary_gpu = torch.cuda.get_device_name(0)

        if REQUIRE_L4_GPU and "L4" not in primary_gpu:
            raise RuntimeError(
                f"Required NVIDIA L4 GPU not available. "
                f"Detected GPU: {primary_gpu}."
            )

        device = "cuda"

    else:
        # CPU fallback keeps the environment state explicit, although later
        # representation processing may be substantially slower without CUDA.
        print("WARNING: No CUDA GPU detected.")
        device = "cpu"

except Exception as e:
    # Any import, CUDA, or hardware-validation failure is reported here and
    # leaves the notebook configured for CPU execution.
    print(f"ERROR: PyTorch not available ({e})")
    device = "cpu"

# ------------------------------------------------------------
# Dependency Verification (Runtime Environment Gate)
# ------------------------------------------------------------

# These packages form the minimum runtime stack required for model execution,
# tensor processing, tabular validation, and transformer-based utilities.
required_packages = [
    "torch",
    "transformers",
    "accelerate",
    "numpy",
    "pandas",
]

print("\nDependency Verification")
print("-" * 60)

# Store structured results in addition to printing them so the final summary can
# distinguish a complete environment from one with missing dependencies.
dependency_status = []

for package_name in required_packages:

    try:
        module = importlib.import_module(package_name)
        version = getattr(module, "__version__", "unknown")

        dependency_status.append({
            "package": package_name,
            "status": "OK",
            "version": version,
        })

        print(f"[OK]   {package_name:<15} {version}")

    except Exception:

        # A failed import is recorded without stopping the loop so every
        # required package is checked in a single verification pass.
        dependency_status.append({
            "package": package_name,
            "status": "MISSING",
            "version": "",
        })

        print(f"[FAIL] {package_name}")

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

# Reduce the detailed dependency results to the package names that require
# attention before later notebook steps can run reliably.
missing_packages = [
    item["package"]
    for item in dependency_status
    if item["status"] != "OK"
]

print("\nVerification Summary")
print("-" * 60)
print(f"Execution Device : {device}")

if len(missing_packages) == 0:
    print("All required dependencies are available.")
else:
    print("Missing packages:")
    for pkg in missing_packages:
        print(f"  - {pkg}")



### 🔷 Step 4 — Load Trained Autoencoder Model

* Locate the trained autoencoder model artifacts for the active experiment.
* Verify that the trained model artifacts exist in the Google Drive experiment directory.
* Import the shared autoencoder model definition from the project source tree.
* Restore the trained model weights from the saved trained model artifacts.
* Move the model to the active computation device.
* Set the autoencoder to evaluation mode for representation workflows.

In [ ]:
# ============================================================
# Step 4: Load Trained Autoencoder Model
# ============================================================

import torch

print("Loading trained autoencoder model...")

# ------------------------------------------------------------
# Checkpoint path from experiment configuration
# ------------------------------------------------------------

# Reuse the checkpoint path established during environment initialization so
# model loading remains tied to the selected Notebook 03 experiment.
CHECKPOINT_PATH = AUTOENCODER_MODEL_PATH

print(f"Checkpoint path:\n{CHECKPOINT_PATH}")

# Validate the upstream artifact before constructing the model, providing a
# clear failure point when the expected trained checkpoint is unavailable.
if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        "Autoencoder checkpoint not found at expected experiment path:\n"
        f"{CHECKPOINT_PATH}"
    )

# ------------------------------------------------------------
# Load model
# ------------------------------------------------------------

# Import the same architecture definition used during training so the saved
# parameter tensors can be restored into a structurally identical model.
from src.autoencoder_model import ConvAutoencoder

# Select CUDA when available, while preserving CPU compatibility for environments
# where a GPU is not present.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Instantiate the autoencoder and place it on the target device before loading
# the trained parameter values.
model = ConvAutoencoder().to(device)

# ------------------------------------------------------------
# Load weights
# ------------------------------------------------------------

# map_location ensures the checkpoint can be restored even when the current
# execution device differs from the device used during training.
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)

# Support both wrapped training checkpoints and files containing only the raw
# model state dictionary.
if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    model.load_state_dict(checkpoint["model_state_dict"])
else:
    model.load_state_dict(checkpoint)

# Switch to evaluation mode so layers behave deterministically during latent
# representation extraction and no training-specific state is updated.
model.eval()

# ------------------------------------------------------------
# Confirm
# ------------------------------------------------------------

print("Autoencoder model loaded successfully.")
print(f"Device: {device}")



### 🔷 Step 5 — Prepare Development Evaluation Dataset

* Select the configured evaluation split from the NExT-QA annotations.
* Sample the configured development subset using the project random seed.
* Validate required video, question, answer, and answer-choice columns.
* Attach the ground-truth answer text for each evaluation record.
* Add representation metadata fields for the active experiment.
* Validate the prepared evaluation dataset before representation alignment.

In [ ]:
# ============================================================
# Step 5: Prepare Development Evaluation Dataset
# ============================================================

from pathlib import Path
import pandas as pd

print("Preparing autoencoder development evaluation subset...")

# Copy the shared experiment settings into local variables so the dataset
# preparation logic clearly reflects the active evaluation configuration.
evaluation_split = EVALUATION_SPLIT
development_subset_size = DEVELOPMENT_SUBSET_SIZE
random_seed = RANDOM_SEED

# ------------------------------------------------------------
# Validate annotation columns
# ------------------------------------------------------------

# Confirm that the combined annotation table contains every field needed to
# select questions, interpret answer labels, and retain all candidate choices.
required_annotation_columns = [
    "split",
    VIDEO_ID_COLUMN,
    QUESTION_COLUMN,
    GROUND_TRUTH_ANSWER_COLUMN,
] + CHOICE_COLUMNS

missing_annotation_columns = [
    col for col in required_annotation_columns
    if col not in annotations_df.columns
]

# Stop before sampling if the annotation schema differs from the project's
# expected NExT-QA structure.
if missing_annotation_columns:
    raise ValueError(
        f"annotations_df is missing required columns: {missing_annotation_columns}"
    )

# ------------------------------------------------------------
# Select evaluation split and development subset
# ------------------------------------------------------------

# Restrict the combined annotations to the configured evaluation partition
# before constructing the smaller development subset.
eval_df = annotations_df[
    annotations_df["split"] == evaluation_split
].copy()

if len(eval_df) == 0:
    raise ValueError(f"No records found for split: {evaluation_split}")

# Cap the requested sample size at the number of available records so the same
# code remains valid for either development-scale or smaller input datasets.
sample_size = min(
    development_subset_size,
    len(eval_df),
)

# Use the configured random seed to make the sampled question set reproducible
# across reruns and comparable with related experiment outputs.
eval_df = (
    eval_df
    .sample(
        n=sample_size,
        random_state=random_seed,
    )
    .reset_index(drop=True)
)

print(f"Selected evaluation samples: {len(eval_df):,}")

# ------------------------------------------------------------
# Attach ground-truth answer text
# ------------------------------------------------------------

# Convert each numeric answer label into the corresponding candidate-answer
# text so later evaluation artifacts contain both forms of the ground truth.
def answer_index_to_text(row):
    answer_idx = int(row[GROUND_TRUTH_ANSWER_COLUMN])
    option_col = f"a{answer_idx}"

    # Validate the derived column name before indexing the row, which protects
    # against malformed or out-of-range answer labels.
    if option_col not in CHOICE_COLUMNS:
        raise ValueError(f"Answer option column not valid: {option_col}")

    return row[option_col]

# Apply the conversion row by row because the correct answer text is selected
# from a different candidate column for each question.
eval_df["ground_truth_text"] = eval_df.apply(
    answer_index_to_text,
    axis=1,
)

# ------------------------------------------------------------
# Add representation source metadata
# ------------------------------------------------------------

# Label every record with its representation type and originating experiment
# so downstream outputs remain traceable when multiple experiments are compared.
eval_df["video_source"] = "autoencoder_latent"
eval_df["representation_experiment"] = EXPERIMENT_NAME

# ------------------------------------------------------------
# Final validation
# ------------------------------------------------------------

# Define the complete schema expected by subsequent representation-validation
# and evaluation steps.
required_eval_columns = [
    VIDEO_ID_COLUMN,
    QUESTION_COLUMN,
    GROUND_TRUTH_ANSWER_COLUMN,
    *CHOICE_COLUMNS,
    "ground_truth_text",
    "video_source",
    "representation_experiment",
]

missing_columns = [
    col for col in required_eval_columns
    if col not in eval_df.columns
]

# Perform a final schema gate after all derived and metadata columns have been
# added, ensuring the prepared dataframe is ready for downstream use.
if missing_columns:
    raise ValueError(
        f"Missing required evaluation columns: {missing_columns}"
    )

print("\nAutoencoder evaluation dataset prepared successfully.")
print(f"Evaluation samples : {len(eval_df):,}")
print(f"Evaluation split   : {evaluation_split}")
print(f"Random seed        : {random_seed}")
print(f"Answer mode        : {ANSWER_MODE}")
print(f"Video source       : autoencoder_latent")

# Display a small sample to confirm the selected questions, answer mapping, and
# representation metadata without printing the entire evaluation dataframe.
print("\nEvaluation Dataset Preview:")
display(eval_df.head())



### 🔷 Step 6 — Load and Standardize Autoencoder Video Representations

* Locate the segment-level and video-level autoencoder representation files generated by Notebook 03.
* Verify that both representation artifacts exist before loading them.
* Load the segment-level and video-level representation datasets.
* Validate the required video identifiers, split information, segment counts, representation metadata, and embedding columns.
* Standardize video identifiers and representation metadata using the shared Notebook 07-compatible schema.
* Confirm that the expected training, validation, and test split values are valid and that the configured evaluation split is available.
* Check for duplicate identifiers and missing embedding values.
* Save the standardized video representation artifact and create compatibility aliases for downstream processing.



In [ ]:
# ============================================================
# Step 6: Load and Standardize Autoencoder Video Representations
# ============================================================

import pandas as pd
import re

print("Loading and standardizing autoencoder latent representations...")

# ------------------------------------------------------------
# Locate Drive-based representation outputs
# ------------------------------------------------------------

# Reuse the experiment-specific artifact paths established by the shared
# configuration so this notebook validates the exact outputs from Notebook 03.
segment_representations_path = AUTOENCODER_SEGMENT_REPRESENTATIONS_CSV
video_representations_path = AUTOENCODER_VIDEO_REPRESENTATIONS_CSV

print(f"Segment representation path:\n{segment_representations_path}")
print(f"\nVideo representation path:\n{video_representations_path}")

# Both artifacts are required: segment-level data supports provenance and
# diagnostics, while video-level vectors are the downstream VideoQA inputs.
if not segment_representations_path.exists():
    raise FileNotFoundError(
        f"Segment representations not found: {segment_representations_path}"
    )

if not video_representations_path.exists():
    raise FileNotFoundError(
        f"Video representations not found: {video_representations_path}"
    )

# Load the generated CSV artifacts into memory for schema, coverage, and value
# validation before they are exposed to later notebooks.
ae_segment_representation_df = pd.read_csv(segment_representations_path)
ae_video_representation_df = pd.read_csv(video_representations_path)

# ------------------------------------------------------------
# Validate required video representation columns
# ------------------------------------------------------------

# These metadata fields establish video identity, dataset membership, segment
# aggregation history, experiment provenance, and representation source.
required_video_columns = [
    "video",
    "split",
    "segment_count",
    "representation_experiment",
    "representation_source",
]

missing_video_columns = [
    col for col in required_video_columns
    if col not in ae_video_representation_df.columns
]

if missing_video_columns:
    raise ValueError(
        f"ae_video_representation_df is missing columns: {missing_video_columns}"
    )

# Identify only embedding columns that follow the project's standardized
# zero-padded naming convention, excluding unrelated numeric metadata fields.
video_embedding_columns = [
    col for col in ae_video_representation_df.columns
    if re.fullmatch(r"embedding_\d{3}", col)
]

if len(video_embedding_columns) == 0:
    raise ValueError("No standardized embedding columns found.")

# Sort the columns so vector dimensions remain in numerical order regardless
# of how the CSV reader or upstream writer arranged them.
video_embedding_columns = sorted(video_embedding_columns)

# The observed vector width must match the latent dimension defined by the
# trained autoencoder architecture.
if len(video_embedding_columns) != AUTOENCODER_LATENT_DIM:
    raise ValueError(
        f"Expected {AUTOENCODER_LATENT_DIM} embedding dimensions, "
        f"found {len(video_embedding_columns)}."
    )

# ------------------------------------------------------------
# Standardize identifiers and metadata
# ------------------------------------------------------------

# Convert identifiers and split labels to strings to avoid inconsistent joins
# caused by mixed numeric and textual types in downstream processing.
ae_video_representation_df["video"] = (
    ae_video_representation_df["video"].astype(str)
)

ae_video_representation_df["split"] = (
    ae_video_representation_df["split"].astype(str)
)

# Create a representation-specific record identifier while retaining the
# original video identifier used to join with NExT-QA annotations.
ae_video_representation_df["record_id"] = (
    "ae_video_" + ae_video_representation_df["video"]
)

# Normalize metadata values to the shared video-representation vocabulary used
# by Notebook 07 and other representation pipelines.
ae_video_representation_df["representation_source"] = "autoencoder_video"
ae_video_representation_df["representation_type"] = "video"
ae_video_representation_df["model_name"] = "conv_autoencoder"
ae_video_representation_df["embedding_dimension"] = AUTOENCODER_LATENT_DIM

# ------------------------------------------------------------
# Validate split values
# ------------------------------------------------------------

# Restrict split labels to the three official NExT-QA partitions so typographical
# errors or unexpected upstream categories cannot silently propagate.
valid_splits = {
    "train",
    "val",
    "test",
}

observed_splits = set(
    ae_video_representation_df["split"]
    .dropna()
    .astype(str)
    .unique()
)

invalid_splits = sorted(observed_splits - valid_splits)

if invalid_splits:
    raise ValueError(
        f"Unexpected split values found in AE video representations: {invalid_splits}"
    )

# Explicitly require the configured evaluation split because downstream
# validation cannot proceed without representations for that partition.
if EVALUATION_SPLIT not in observed_splits:
    raise ValueError(
        f"Evaluation split '{EVALUATION_SPLIT}' was not found in AE video representations. "
        "Rerun Notebook 03 Step 12 to regenerate train/val/test AE representations."
    )

# ------------------------------------------------------------
# Reorder columns to match shared video representation structure
# ------------------------------------------------------------

# Place descriptive metadata first and the latent vector dimensions afterward
# to match the common schema expected by downstream representation workflows.
metadata_columns = [
    "record_id",
    "video",
    "split",
    "representation_source",
    "representation_type",
    "representation_experiment",
    "model_name",
    "embedding_dimension",
    "segment_count",
]

ae_video_representation_df = ae_video_representation_df[
    metadata_columns + video_embedding_columns
]

# ------------------------------------------------------------
# Validate standardized schema
# ------------------------------------------------------------

# Every standardized representation row must have a unique record identifier.
duplicate_record_ids = ae_video_representation_df["record_id"].duplicated().sum()

if duplicate_record_ids > 0:
    raise ValueError(
        f"Found {duplicate_record_ids} duplicate autoencoder record_id values."
    )

# Each video should contribute exactly one aggregated video-level representation.
duplicate_video_ids = ae_video_representation_df["video"].duplicated().sum()

if duplicate_video_ids > 0:
    raise ValueError(
        f"Found {duplicate_video_ids} duplicate autoencoder video values."
    )

# Missing latent values would make vector operations and model inference
# invalid, so the complete embedding matrix is checked before saving.
missing_embedding_values = (
    ae_video_representation_df[video_embedding_columns]
    .isna()
    .sum()
    .sum()
)

if missing_embedding_values > 0:
    raise ValueError(
        f"Found {missing_embedding_values} missing embedding values."
    )

# ------------------------------------------------------------
# Save standardized video representation artifact
# ------------------------------------------------------------

# Overwrite the video-level artifact with the validated shared schema so later
# notebooks can consume it directly without repeating this normalization step.
ae_video_representation_df.to_csv(
    video_representations_path,
    index=False,
)

# ------------------------------------------------------------
# Notebook 07 compatibility aliases
# ------------------------------------------------------------

# Provide the dataframe names expected by the representation-based VideoQA
# workflow while preserving the validated autoencoder dataframe unchanged.
standardized_video_representation_df = ae_video_representation_df.copy()
filtered_video_representation_df = ae_video_representation_df.copy()

# ------------------------------------------------------------
# Final summary
# ------------------------------------------------------------

print("\nAutoencoder latent representations loaded and standardized successfully.")
print(f"Segment representations : {len(ae_segment_representation_df):,}")
print(f"Video representations   : {len(ae_video_representation_df):,}")
print(f"Embedding dimensions    : {len(video_embedding_columns):,}")
print(f"Representation source   : autoencoder_video")
print(f"Experiment              : {EXPERIMENT_NAME}")

print("\nVideo Representation Split Counts")
print("-" * 60)

# Summarize representation coverage by dataset split so missing or imbalanced
# partitions are visible before downstream evaluation begins.
display(
    ae_video_representation_df
    .groupby("split")
    .size()
    .rename("video_count")
    .reset_index()
)

# Show metadata and only the first few vector dimensions to verify structure
# without rendering the complete high-dimensional representation table.
print("\nStandardized Video Representation Preview:")
display(
    ae_video_representation_df[
        [
            "record_id",
            "video",
            "split",
            "representation_source",
            "representation_type",
            "representation_experiment",
            "model_name",
            "embedding_dimension",
            "segment_count",
            *video_embedding_columns[:5],
        ]
    ].head()
)



### 🔷 Step 7 — Validate Autoencoder Representation Dataset

* Verify that the segment-level and standardized video-level representation datasets were loaded successfully.
* Confirm that the video representation dataset follows the shared Notebook 07-compatible schema.
* Validate representation identifiers, split information, source metadata, embedding dimensions, segment counts, and latent feature columns.
* Check for missing or nonnumeric embedding values.
* Detect zero segment counts and duplicate video records.
* Generate and display a validation summary.
* Confirm that the autoencoder video representations are ready for selection and merging in Notebook 07.


In [ ]:
# ============================================================
# Step 7: Validate Autoencoder Representation Dataset
# ============================================================

import pandas as pd

print("Validating autoencoder representation dataset...")

# ------------------------------------------------------------
# Verify required objects from Step 6
# ------------------------------------------------------------

# Confirm that the standardized representation tables and embedding-column list
# were created before this validation step is allowed to continue.
required_objects = [
    "ae_video_representation_df",
    "ae_segment_representation_df",
    "video_embedding_columns",
]

missing_objects = [
    obj for obj in required_objects
    if obj not in globals()
]

if missing_objects:
    raise NameError(
        f"Missing required objects from Step 6: {missing_objects}"
    )

# ------------------------------------------------------------
# Validate Notebook 07 / CLIP-compatible schema
# ------------------------------------------------------------

# Validate the complete shared schema expected by Notebook 07, including
# identifiers, provenance metadata, aggregation information, and latent values.
required_video_columns = [
    "record_id",
    VIDEO_ID_COLUMN,
    "split",
    "representation_source",
    "representation_type",
    "representation_experiment",
    "model_name",
    "embedding_dimension",
    "segment_count",
    *video_embedding_columns,
]

missing_video_columns = [
    col for col in required_video_columns
    if col not in ae_video_representation_df.columns
]

if missing_video_columns:
    raise ValueError(
        f"ae_video_representation_df is missing columns required by "
        f"Notebook 07 representation schema: {missing_video_columns}"
    )

# Require a single, standardized source label so downstream workflows can
# distinguish autoencoder vectors from CLIP and hybrid representations.
expected_representation_source = "autoencoder_video"

representation_sources = (
    ae_video_representation_df["representation_source"]
    .dropna()
    .unique()
    .tolist()
)

if representation_sources != [expected_representation_source]:
    raise ValueError(
        f"Expected representation_source='{expected_representation_source}', "
        f"found {representation_sources}."
    )

# ------------------------------------------------------------
# Validate embedding values
# ------------------------------------------------------------

# Check the latent matrix for incomplete values that would invalidate vector
# operations or downstream classifier inputs.
missing_embedding_values = (
    ae_video_representation_df[video_embedding_columns]
    .isna()
    .sum()
    .sum()
)

# Every embedding dimension must use a numeric dtype so the representation can
# be converted directly into tensors or numerical arrays.
non_numeric_embedding_columns = [
    col for col in video_embedding_columns
    if not pd.api.types.is_numeric_dtype(ae_video_representation_df[col])
]

# Each video-level vector must be based on at least one segment; a nonpositive
# count indicates an invalid or incomplete aggregation result.
zero_segment_count = (
    ae_video_representation_df["segment_count"] <= 0
).sum()

# A video should appear only once because each row represents its single
# aggregated video-level latent representation.
duplicate_video_count = (
    ae_video_representation_df[VIDEO_ID_COLUMN]
    .duplicated()
    .sum()
)

# Inspect the declared dimension metadata independently from the actual number
# of embedding columns so both structural descriptions can be cross-checked.
embedding_dimension_values = (
    ae_video_representation_df["embedding_dimension"]
    .dropna()
    .unique()
    .tolist()
)

# ------------------------------------------------------------
# Build validation summary
# ------------------------------------------------------------

# Consolidate the primary schema, coverage, and data-quality checks into a
# compact table that can be reviewed before strict failures are evaluated.
validation_summary = {
    "video_representations_loaded": len(ae_video_representation_df),
    "unique_videos": ae_video_representation_df[VIDEO_ID_COLUMN].nunique(),
    "segment_representations_loaded": len(ae_segment_representation_df),
    "embedding_dimensions": len(video_embedding_columns),
    "schema_compatible_with_notebook_07": True,
    "representation_source": expected_representation_source,
    "missing_embedding_values": int(missing_embedding_values),
    "non_numeric_embedding_columns": len(non_numeric_embedding_columns),
    "zero_segment_count": int(zero_segment_count),
    "duplicate_video_records": int(duplicate_video_count),
    "answer_mode": ANSWER_MODE,
    "representation_experiment": EXPERIMENT_NAME,
}

validation_df = pd.DataFrame(
    validation_summary.items(),
    columns=["Validation Check", "Value"],
)

display(validation_df)

# ------------------------------------------------------------
# Raise errors for failed validation checks
# ------------------------------------------------------------

# Compare the physical vector width with the latent dimension defined by the
# autoencoder architecture.
if len(video_embedding_columns) != AUTOENCODER_LATENT_DIM:
    raise ValueError(
        f"Expected {AUTOENCODER_LATENT_DIM} embedding dimensions, "
        f"found {len(video_embedding_columns)}."
    )

# Confirm that the dimension recorded in every row agrees with the actual
# standardized embedding-column count.
if embedding_dimension_values != [AUTOENCODER_LATENT_DIM]:
    raise ValueError(
        f"Expected embedding_dimension={AUTOENCODER_LATENT_DIM}, "
        f"found {embedding_dimension_values}."
    )

if missing_embedding_values > 0:
    raise ValueError(
        f"Found {missing_embedding_values} missing embedding values."
    )

if non_numeric_embedding_columns:
    raise ValueError(
        f"Non-numeric embedding columns found: {non_numeric_embedding_columns[:10]}"
    )

if zero_segment_count > 0:
    raise ValueError(
        f"Found {zero_segment_count} records with zero segment count."
    )

if duplicate_video_count > 0:
    raise ValueError(
        f"Found {duplicate_video_count} duplicate video records."
    )

# Reaching this point confirms that the autoencoder artifact can be consumed by
# the same downstream interface used for other video representation sources.
print(
    "\nRepresentation validation passed. "
    "Autoencoder video representations follow the Notebook 07 / CLIP-compatible schema."
)



### 🔷 Step 8 — Verify Representation Artifacts

* Verify that the segment-level and video-level autoencoder representation objects are available.
* Confirm the loaded segment and video representation counts.
* Report the standardized embedding dimensionality and representation source.
* Confirm that the video representation artifact follows the shared `embedding_###` column convention.
* Verify readiness for Notebook 07, where the representations will be merged with annotations and shared CLIP text embeddings.



In [ ]:
# ============================================================
# Step 8: Verify Representation Artifacts
# ============================================================

print("Verifying representation artifacts...")

# Confirm that the validated representation tables and embedding-column list
# remain available before handing the artifacts off to downstream notebooks.
required_objects = [
    "ae_video_representation_df",
    "ae_segment_representation_df",
    "video_embedding_columns",
]

missing_objects = [
    obj for obj in required_objects
    if obj not in globals()
]

# Fail clearly if this step is run out of sequence or a required artifact was
# not created successfully by the preceding preparation steps.
if missing_objects:
    raise NameError(
        f"Missing representation artifacts: {missing_objects}"
    )

print("Representation artifacts verified.")

# Report the final artifact sizes and vector width as a concise handoff summary.
print(f"Video representations   : {len(ae_video_representation_df):,}")
print(f"Segment representations : {len(ae_segment_representation_df):,}")
print(f"Embedding dimensions    : {len(video_embedding_columns):,}")

# Clarify the conceptual transition from representation preparation in this
# notebook to multimodal VideoQA assembly in Notebook 07.
print("\nNotebook 07 will merge these representations with")
print("the evaluation dataset and shared CLIP text embeddings.")

# Display the standardized source label and embedding-column range expected by
# the shared downstream representation interface.
print("\nShared representation schema:")
print(f"Representation source   : autoencoder_video")
print(f"Embedding columns       : {video_embedding_columns[0]} ... {video_embedding_columns[-1]}")



### 🔷 Step 9 — Generate Autoencoder Representation Summary Report

* Generate summary statistics for the loaded autoencoder representation artifacts.
* Report the experiment name, representation source and type, model name, and answer mode.
* Summarize segment-level and video-level representation counts and the number of unique videos.
* Record the observed and expected embedding dimensionality.
* Calculate minimum, maximum, and mean segment counts across represented videos.
* Report the number of missing embedding values.
* If Google Drive writes are enabled, save the summary report to the active autoencoder representation directory.
* Otherwise, display the summary report without writing it to Google Drive.



In [ ]:
# ============================================================
# Step 9: Generate Autoencoder Representation Summary Report
# ============================================================

import pandas as pd

print("Generating autoencoder representation summary report...")

# Confirm that the validated representation artifacts are available before
# computing and saving the final notebook-level summary.
required_objects = [
    "ae_video_representation_df",
    "ae_segment_representation_df",
    "video_embedding_columns",
]

missing_objects = [
    obj for obj in required_objects
    if obj not in globals()
]

# This guard prevents an incomplete summary from being produced when the
# representation-loading step has not run successfully.
if missing_objects:
    raise NameError(
        f"Missing required objects from Step 6: {missing_objects}"
    )

# ------------------------------------------------------------
# Build summary rows
# ------------------------------------------------------------

# Assemble experiment identity, representation coverage, vector dimensions,
# segment aggregation statistics, and embedding completeness into one report.
summary_rows = [
    {"metric": "experiment_name", "value": EXPERIMENT_NAME},
    {"metric": "experiment_type", "value": "autoencoder_representation"},
    {"metric": "representation_source", "value": "autoencoder_video"},
    {"metric": "representation_type", "value": "video"},
    {"metric": "model_name", "value": "conv_autoencoder"},
    {"metric": "answer_mode", "value": ANSWER_MODE},
    {"metric": "video_representations", "value": len(ae_video_representation_df)},
    {"metric": "unique_videos", "value": ae_video_representation_df[VIDEO_ID_COLUMN].nunique()},
    {"metric": "segment_representations", "value": len(ae_segment_representation_df)},
    {"metric": "embedding_dimensions", "value": len(video_embedding_columns)},
    {"metric": "expected_embedding_dimensions", "value": AUTOENCODER_LATENT_DIM},
    {"metric": "minimum_segment_count", "value": int(ae_video_representation_df["segment_count"].min())},
    {"metric": "maximum_segment_count", "value": int(ae_video_representation_df["segment_count"].max())},
    {"metric": "mean_segment_count", "value": round(ae_video_representation_df["segment_count"].mean(), 2)},
    {
        "metric": "missing_embedding_values",
        "value": int(
            ae_video_representation_df[video_embedding_columns]
            .isna()
            .sum()
            .sum()
        ),
    },
]

# Convert the metric records into a simple two-column table that is easy to
# inspect manually and consume programmatically in later reporting workflows.
autoencoder_summary_df = pd.DataFrame(summary_rows)

# ------------------------------------------------------------
# Save summary report (Optional)
# ------------------------------------------------------------

if ENABLE_GOOGLE_DRIVE_WRITES:

    # Ensure the experiment's Drive-based representation directory exists before
    # writing the persistent summary artifact.
    AUTOENCODER_REPRESENTATIONS_DRIVE_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    autoencoder_summary_csv = (
        AUTOENCODER_REPRESENTATIONS_DRIVE_DIR
        / "autoencoder_representation_summary.csv"
    )

    # Save without a dataframe index so the CSV contains only the intended metric
    # and value fields.
    autoencoder_summary_df.to_csv(
        autoencoder_summary_csv,
        index=False,
    )

    print("Autoencoder representation summary report saved.")
    print(f"Summary file : {autoencoder_summary_csv}")

else:

    print("Google Drive writes are disabled.")
    print("Autoencoder representation summary report was not saved.")

# Display the same report in the notebook to provide an immediate final review
# of the prepared autoencoder representation artifacts.
display(autoencoder_summary_df)



### 🔷 Step 10 — Display Sample Representation Records

* Randomly sample standardized autoencoder video representation records for inspection.
* Display video identifiers, segment counts, representation metadata, and the first five embedding dimensions.
* Report the total number of video representations and unique videos.
* Summarize the embedding dimensionality and segment-count statistics.
* Support qualitative verification of the standardized video representation artifact.


In [ ]:
# ============================================================
# Step 10: Display Sample Representation Records
# ============================================================

import pandas as pd

# Confirm that the validated video representations and standardized embedding
# column list are available before generating an inspection sample.
required_objects = [
    "ae_video_representation_df",
    "video_embedding_columns",
]

missing_objects = [
    obj for obj in required_objects
    if obj not in globals()
]

if missing_objects:
    raise NameError(
        f"Missing required objects from Step 6: {missing_objects}"
    )

# Limit the preview to a manageable number of records while automatically
# adapting to smaller development datasets.
sample_count = min(
    10,
    len(ae_video_representation_df),
)

# Select a reproducible random sample so repeated notebook executions display
# the same records when the random seed is unchanged.
sample_representation_df = (
    ae_video_representation_df
    .sample(
        n=sample_count,
        random_state=RANDOM_SEED,
    )
    .reset_index(drop=True)
)

# Show descriptive metadata together with only the first few latent dimensions,
# keeping the preview readable despite the high-dimensional embeddings.
display_columns = [
    VIDEO_ID_COLUMN,
    "segment_count",
    "representation_source",
    "representation_type",
    "representation_experiment",
    *video_embedding_columns[:5],
]

print(f"Displaying {sample_count} autoencoder video representations...")
print("Representation source : autoencoder_video")
print(f"Embedding dimensions  : {len(video_embedding_columns)}")
print("\nShowing first 5 embedding dimensions only.\n")

display(
    sample_representation_df[
        display_columns
    ]
)

# ------------------------------------------------------------
# Representation Summary
# ------------------------------------------------------------

# Conclude the notebook with a concise summary of representation coverage,
# aggregation statistics, and latent-vector dimensionality.
print("\nRepresentation Summary")
print("-" * 60)
print(f"Video representations   : {len(ae_video_representation_df):,}")
print(f"Unique videos           : {ae_video_representation_df[VIDEO_ID_COLUMN].nunique():,}")
print(f"Embedding dimensions    : {len(video_embedding_columns):,}")
print(f"Min segment count       : {int(ae_video_representation_df['segment_count'].min())}")
print(f"Max segment count       : {int(ae_video_representation_df['segment_count'].max())}")
print(f"Mean segment count      : {ae_video_representation_df['segment_count'].mean():.2f}")



### 🔷 Step 11 — Notebook Summary

* Summarize the completed autoencoder representation loading, standardization, and validation workflow.
* Report representation counts for the training, validation, and test splits.
* Display segment-level and video-level representation statistics and embedding dimensionality.
* Identify the standardized representation files and generated summary report.
* Confirm compatibility with the shared Notebook 07 representation schema.
* Describe how Notebook 07 will combine the autoencoder video representations with NExT-QA annotations and shared CLIP text embeddings.



In [ ]:
# ============================================================
# Step 11: Notebook Summary
# ============================================================

print("Notebook 04 complete.")
print("=" * 60)

# Aggregate the validated video representations by dataset split so the final
# report can show the exact train, validation, and test coverage.
split_counts = (
    ae_video_representation_df
    .groupby("split")
    .size()
    .to_dict()
)

# Summarize the active experiment configuration to preserve the context under
# which these representation artifacts were prepared and validated.
print("\nAutoencoder Representation Experiment")
print("-" * 60)
print(f"Experiment name          : {EXPERIMENT_NAME}")
print(f"Training split           : train")
print(f"Evaluation split         : {EVALUATION_SPLIT}")
print(f"Development subset size  : {DEVELOPMENT_SUBSET_SIZE}")
print(f"Answer mode              : {ANSWER_MODE}")

# Report representation coverage at both the dataset-split level and the
# aggregate level, together with the latent-vector dimensionality.
print("\nRepresentation Dataset")
print("-" * 60)
print(f"Video representations    : {len(ae_video_representation_df):,}")
print(f"  Training videos        : {split_counts.get('train', 0):,}")
print(f"  Validation videos      : {split_counts.get('val', 0):,}")
print(f"  Test videos            : {split_counts.get('test', 0):,}")
print(f"Unique videos            : {ae_video_representation_df[VIDEO_ID_COLUMN].nunique():,}")
print(f"Segment representations  : {len(ae_segment_representation_df):,}")
print(f"Embedding dimensions     : {len(video_embedding_columns):,}")

# Segment-count statistics describe how many temporal units contributed to each
# aggregated video representation and expose the range of video coverage.
print("\nRepresentation Statistics")
print("-" * 60)
print(f"Minimum segment count    : {int(ae_video_representation_df['segment_count'].min())}")
print(f"Maximum segment count    : {int(ae_video_representation_df['segment_count'].max())}")
print(f"Mean segment count       : {ae_video_representation_df['segment_count'].mean():.2f}")

# List the persistent files and directory produced or standardized by this
# notebook so downstream users can locate the complete representation package.
print("\nGenerated Outputs")
print("-" * 60)
print(f"Video representations    : {AUTOENCODER_VIDEO_REPRESENTATIONS_CSV.name}")
print(f"Segment representations  : {AUTOENCODER_SEGMENT_REPRESENTATIONS_CSV.name}")
print(f"Representation summary   : {autoencoder_summary_csv.name}")
print(f"Representation directory : {AUTOENCODER_REPRESENTATIONS_DRIVE_DIR}")

# Describe the logical artifacts created during the workflow, including both
# persistent files and notebook-level validation evidence.
print("\nGenerated Artifacts")
print("-" * 60)
print("- Autoencoder video representations (train, validation, test)")
print("- Autoencoder segment representations (train, validation, test)")
print("- Autoencoder representation summary")
print("- Schema validation report")
print("- Sample representation records")

# Conclude with the handoff contract between this preparation notebook and the
# shared representation-based VideoQA workflow in Notebook 07.
print("\nCompatibility")
print("-" * 60)
print("Representation artifacts follow the shared Notebook 07 schema.")
print("Video embeddings use standardized embedding_### columns.")
print("Notebook 07 will select the required training and validation")
print("representations, merge them with the corresponding NExT-QA")
print("annotations and shared CLIP text embeddings, and train/evaluate")
print("the Fusion MLP classifier.")

